# ML Pipeline Governance Framework

## Pipelines, Tuning, and Governance

## Role

You are a machine learning engineer responsible for governance at a SaaS company.

## Scenario

An internal audit requires proof that preprocessing, tuning, and evaluation follow **reproducible and policy-compliant workflows**. You must demonstrate that your modeling process avoids data leakage, enforces separation of concerns, and supports transparent governance through pipelines.

Your objective is to show how **pipelines, controlled tuning, and disciplined evaluation** ensure that models are trustworthy, auditable, and production-ready.


## Data Source (Required)

Use a file-based dataset loaded from a link.

**Credit Risk Dataset (CSV):**
[https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv](https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv)


## Question 1 — Conceptual: Why Tuning Must Not Involve Test Data

### Teaching Focus (Read Before Answering)

Hyperparameter tuning is a form of **model selection**. If the test set is used during tuning, it becomes part of the training process, and the final evaluation is no longer unbiased.

**Answer:**
*(Tuning must not use the test set because it would cause data leakage and the model would indirectly learn patterns form the test data, making our final evaluation baised and not true measure of performance.)*

## Question 2 — Conceptual: How Pipelines Enforce Governance

### Teaching Focus (Read Before Answering)

Pipelines formalize the entire modeling process into a single object that can be versioned, audited, and reproduced.

**Answer:**
*(Pipeines enforce governance by combining preprocessing and modelling into one workflow. this avoid data leakage and makes the process easy.)*

## Python: Build a Full Preprocessing + Model Pipeline


In [2]:
# This code demonstrates how to load a dataset from a URL, identify the target column robustly,
# split the dataset into features and target, and then perform a train/validation/test split.
# It uses stratified splitting for classification tasks with a limited number of classes.
# This is a foundational step in preparing data for machine learning model training and evaluation.

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

url = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
df = pd.read_csv(url)

# Identify target column
possible_targets = ["credit_risk", "CreditRisk", "Risk", "Class", "default", "Default", "target", "Target"]
target_col = None
for c in possible_targets:
    if c in df.columns:
        target_col = c
        break
if target_col is None:
    target_col = df.columns[-1]

# Split dataset into features (X) and target (y)
X = df.drop(columns=[target_col])  # Features: all columns except target
y = df[target_col]  # Target variable

# Perform train/validation/test split with 60/20/20 ratio

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE,
    stratify=y if y.nunique() <= 20 else None
)
# Second split: split temporary set equally into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE,
    stratify=y_temp if y_temp.nunique() <= 20 else None
)

X_train.shape, X_val.shape, X_test.shape

((600, 20), (200, 20), (200, 20))

**Pipeline Construction and Training**

In [3]:


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Detect numeric vs categorical columns
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns

# Preprocessing for numeric features
numeric_transformer = Pipeline(
    steps=[  # # Define steps for numeric pipeline
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]
)

# Preprocessing for categorical features
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocess = ColumnTransformer(  # # Initialize ColumnTransformer to apply different preprocessors
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# Full pipeline
pipeline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

# Train model
pipeline.fit(X_train, y_train)

# Evaluate
train_acc = pipeline.score(X_train, y_train)
val_acc = pipeline.score(X_val, y_val)

train_acc, val_acc

(0.795, 0.74)


## Question 4 — Python: Perform Controlled Hyperparameter Tuning

### Teaching Focus

Tuning must operate **only on training/validation data**, never on the test set.

**Answer**

In [4]:


from sklearn.model_selection import GridSearchCV  # Import GridSearchCV for hyperparameter tuning

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["liblinear", "lbfgs"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,  # Number of cross-validation folds
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['duration', 'amount', 'installment_rate', 'present_residence', 'age',
       'number_credits', 'people_liable'],
      dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy...
                                                                         Index(['status', 'credit_history', 'purpose', 'savings', 'employment_duration',
       'personal_status_sex', 'other_debtors', 'property',
       'other_installment_plans', 'housing', 'job', 'telephone',
       'foreign_worker'],
      dtype='object'))])),
                                       ('model',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=-1,
             param_grid={'model__C': [0.01, 0.1, 1, 10],
                         'model__solver': ['liblinear', 'lbfgs']},
             scoring='accuracy')


## Question 5 — Python: Report Best Parameters and Validation Results

### Teaching Focus

Auditable systems must record **both parameters and performance**.

**Answer**

In [5]:


best_params = grid.best_params_
best_val_score = grid.best_score_

best_params, best_val_score

({'model__C': 0.1, 'model__solver': 'lbfgs'}, np.float64(0.7383333333333334))


## Question 6 — Python: Evaluate Final Performance on Test Set

### Teaching Focus

The test set is used **once**, after tuning is complete.

**Answer**

In [12]:


best_model = grid.best_estimator_

test_acc = best_model.score(X_test, y_test)
print(test_acc*100, "% Test Accuracy")

77.0 % Test Accuracy
